In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd "/content/drive/MyDrive/TCC2"

In [ ]:
!ls -la
!ls -la "tcc_fotos/dataset_tratado"
!ls -la "yolo11n_cls_experimento"

In [ ]:
!pip install -q -r "yolo11n_cls_experimento/requirements.txt"

In [ ]:
!python "yolo11n_cls_experimento/00_conferir_imagens_tratadas.py"

In [ ]:
!python "yolo11n_cls_experimento/01_preparar_dataset.py" --overwrite

In [ ]:
!ls -la "yolo11n_cls_experimento/resultados/dataset_yolo_cls"
!ls -la "yolo11n_cls_experimento/resultados/dataset_yolo_cls/train"
!ls -la "yolo11n_cls_experimento/resultados/dataset_yolo_cls/val"
!ls -la "yolo11n_cls_experimento/resultados/dataset_yolo_cls/test"

In [ ]:
!python "yolo11n_cls_experimento/02_treinar_yolo11n_cls.py" --modelo "yolo11n-cls.pt"

In [ ]:
!python "yolo11n_cls_experimento/03_avaliar_yolo11n_cls.py"

In [ ]:
!python "yolo11n_cls_experimento/04_gerar_figuras_tcc.py"

In [ ]:
!ls -la "yolo11n_cls_experimento/resultados/resultados_tcc"
!ls -la "yolo11n_cls_experimento/resultados/figuras_tcc"

Testar modelo em imagens aleatórias

In [ ]:
!pip install -q ultralytics opencv-python-headless

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from ultralytics import YOLO

# Caminho do melhor modelo treinado
caminho_modelo = "/content/drive/MyDrive/TCC2/yolo11n_cls_experimento/resultados/resultados_tcc/treino_yolo_cls/weights/best.pt"

if not os.path.exists(caminho_modelo):
    raise FileNotFoundError(f"Modelo não encontrado em: {caminho_modelo}")

print("Carregando modelo YOLO-cls...")
modelo = YOLO(caminho_modelo)
print("Modelo carregado com sucesso!\n")


def crop_black_borders(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        return img[y:y+h, x:x+w]

    return img


def remove_specular_reflections(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    return cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)


def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


def tratar_imagem(img_bgr):
    img_bgr = crop_black_borders(img_bgr)
    img_bgr = remove_specular_reflections(img_bgr)
    img_bgr = apply_clahe(img_bgr)
    return img_bgr


print("Faça o upload de uma ou mais imagens laparoscópicas para testar:")
uploaded = files.upload()

pasta_temp = Path("/content/imagens_tratadas_yolo")
pasta_temp.mkdir(exist_ok=True)

for arquivo in uploaded.keys():
    img_original_bgr = cv2.imread(arquivo)

    if img_original_bgr is None:
        print(f"Não foi possível ler a imagem: {arquivo}")
        continue

    img_tratada_bgr = tratar_imagem(img_original_bgr)

    img_original_rgb = cv2.cvtColor(img_original_bgr, cv2.COLOR_BGR2RGB)
    img_tratada_rgb = cv2.cvtColor(img_tratada_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_original_rgb)
    plt.title("Imagem original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_tratada_rgb)
    plt.title("Imagem tratada")
    plt.axis("off")

    plt.show()

    caminho_tratada = pasta_temp / f"{Path(arquivo).stem}_tratada.png"
    cv2.imwrite(str(caminho_tratada), img_tratada_bgr)

    resultado = modelo.predict(
        source=str(caminho_tratada),
        imgsz=224,
        verbose=False
    )[0]

    nomes = resultado.names
    probabilidades = resultado.probs.data.cpu().numpy()

    classe_para_indice = {nome: indice for indice, nome in nomes.items()}

    if "endometriosis" not in classe_para_indice:
        print("Classe 'endometriosis' não encontrada no modelo.")
        print("Classes encontradas:", nomes)
        continue

    indice_endometriose = classe_para_indice["endometriosis"]
    prob_endometriose = float(probabilidades[indice_endometriose])

    indice_predito = int(resultado.probs.top1)
    classe_predita_nome = nomes[indice_predito]
    confianca_predita = float(resultado.probs.top1conf)

    if classe_predita_nome == "endometriosis":
        classe_final = "Com endometriose"
    else:
        classe_final = "Sem endometriose"

    print(f"Arquivo: {arquivo}")
    print(f"Classificação do modelo: {classe_final}")
    print(f"Classe interna prevista: {classe_predita_nome}")
    print(f"Confiança da classe prevista: {confianca_predita:.2%}")
    print(f"Probabilidade estimada para endometriose: {prob_endometriose:.2%}")
    print("-" * 60)